# 缺失值与异常值处理

**面试回答：**缺失和异常首先是业务语义，不是纯数值问题；统计填补只在训练数据拟合，并保留缺失指示与裁剪日志。

## 真实案例

授信申请的收入可能缺失，极端金额可能是大客户或单位错误。

In [1]:
import numpy as np  # 导入 NumPy 手写清洗。
app=np.array(['A01','A02','A03','A04','A05','A06','V01','V02'])  # 构造申请编号。
income=np.array([8.,10.,np.nan,12.,15.,1000.,np.nan,11.])  # 记录收入万元并含缺失和异常。
y=np.array([0,0,1,0,1,1,1,0])  # 记录人工复核标签。
print('申请 | 收入万 | 复核')  # 输出原始字段表头。
for a,b,c in zip(app,income,y):  # 展示样本。
    print(a,b,c)  # 输出申请。

申请 | 收入万 | 复核
A01 8.0 0
A02 10.0 0
A03 nan 1
A04 12.0 0
A05 15.0 1
A06 1000.0 1
V01 nan 1
V02 11.0 0


## Baseline / 基线

错误基线把缺失收入直接填 0，等同于错误声明未授权用户没有收入。

In [2]:
bad=np.nan_to_num(income,nan=0.0)  # 错误地把缺失直接当零。
print('错误填补=',bad.tolist())  # 输出错误基线。
print('缺失条数=',int(np.isnan(income).sum()))  # 输出缺失规模。

错误填补= [8.0, 10.0, 0.0, 12.0, 15.0, 1000.0, 0.0, 11.0]
缺失条数= 2


In [3]:
train=np.arange(6)  # 定义训练申请。
valid=np.arange(6,8)  # 定义验证申请。
median=np.nanmedian(income[train])  # 只在训练集拟合中位数。
missing=np.isnan(income)  # 生成缺失指示列。
filled=np.where(missing,median,income)  # 用训练中位数填补缺失。
cap=np.quantile(filled[train],.9)  # 用训练分位数定义异常上限。
clipped=np.minimum(filled,cap)  # 裁剪超出上限的异常值。
feature=np.c_[clipped,missing.astype(float)]  # 拼接清洗值和缺失指示。
print('训练中位数/上限=',median,round(float(cap),2))  # 输出拟合统计量。
print('清洗特征 [收入,缺失]=',np.round(feature,2).tolist())  # 输出中间特征。

训练中位数/上限= 12.0 507.5
清洗特征 [收入,缺失]= [[8.0, 0.0], [10.0, 0.0], [12.0, 1.0], [12.0, 0.0], [15.0, 0.0], [507.5, 0.0], [12.0, 1.0], [11.0, 0.0]]


## 结果解读

中位数代表训练分布中心，缺失指示保留流程信息；裁剪并不删除原始记录，原始值和裁剪比例需要审计。

In [4]:
rule=(feature[valid,0]>=median)|(feature[valid,1]==1)  # 用清洗后的简单规则产生复核建议。
print('验证申请/建议=',list(zip(app[valid].tolist(),rule.astype(int).tolist())))  # 输出处理结果。
print('生产差距：需字段血缘、缺失原因、异常工单和训练—服务统计量版本。')  # 说明生产边界。
print('不要静默删除极端值，应先确认单位与业务类型。')  # 输出治理原则。

验证申请/建议= [('V01', 1), ('V02', 0)]
生产差距：需字段血缘、缺失原因、异常工单和训练—服务统计量版本。
不要静默删除极端值，应先确认单位与业务类型。


## 失败案例与修复

在全量样本上计算中位数会把未来信息泄回训练；修复是 split 后只在训练段 fit。

In [5]:
global_median=np.nanmedian(income)  # 错误地在全量申请上计算中位数。
print('失败全量中位数=',global_median)  # 输出泄漏统计量。
print('修复训练中位数=',median)  # 输出安全统计量。
print('异常裁剪数=',int(np.sum(filled>cap)))  # 输出可监控的裁剪事件。
print('模型可用缺失分支也不免除语义审计。')  # 说明边界。

失败全量中位数= 11.5
修复训练中位数= 12.0
异常裁剪数= 1
模型可用缺失分支也不免除语义审计。


In [6]:
assert len(app)>=5  # 保护样本数。
assert missing.sum()==2  # 保护缺失指示正确。
assert clipped[5]<=cap  # 保护异常裁剪生效。
assert global_median!=median  # 保护全量统计泄漏反例。